# Phase 1 and 2 - Data Acquisition and Preparation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [2]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ROOT,
    RAW,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    UA
)

print("Project root :", ROOT)
print("Raw data     :", RAW)
print("CFPB         :", CFPB_RAW)
print("eCFR         :", ECFR_RAW)
print("PDF          :", PDF_RAW)
print("Processed    :", PROC)

assert CFPB_RAW.exists()
assert ECFR_RAW.exists()
assert PDF_RAW.exists()

print("\nPASS: Phase 1 directories are ready.")

Project root : /home/sagemaker-user/complaintiq
Raw data     : /home/sagemaker-user/complaintiq/data/raw
CFPB         : /home/sagemaker-user/complaintiq/data/raw/cfpb
eCFR         : /home/sagemaker-user/complaintiq/data/raw/ecfr
PDF          : /home/sagemaker-user/complaintiq/data/raw/pdf
Processed    : /home/sagemaker-user/complaintiq/data/processed

PASS: Phase 1 directories are ready.


In [3]:
download_script = PROJECT_ROOT / "src" / "download_data.py"

print("File:", download_script)
print("=" * 80)

print(download_script.read_text(encoding="utf-8"))

File: /home/sagemaker-user/complaintiq/src/download_data.py



In [4]:
from pathlib import Path

script = r'''from __future__ import annotations

import csv
import io
import re
import zipfile
from datetime import datetime
from html.parser import HTMLParser
from urllib.parse import urljoin

import requests

from src.config import ROOT, CFPB_RAW, UA

ARCHIVE_DIR = ROOT / "data" / "raw" / "cfpb_archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PAGE = (
    "https://www.consumerfinance.gov/foia-requests/"
    "foia-electronic-reading-room/"
    "cfpb-consumer-complaint-database-narratives-archive/"
)

EXPORTS = {
    "CCDB Export January 2026 through February 2026":
        "january_february_2026.zip",
    "CCDB Export March 2026":
        "march_2026.zip",
    "CCDB Export April 2026":
        "april_2026.zip",
    "CCDB Export May 2026":
        "may_2026.zip",
    "CCDB Export June 2026":
        "june_2026.zip",
}

START_DATE = datetime(2026, 1, 1).date()
END_DATE = datetime(2026, 6, 30).date()

PRODUCT_OUTPUTS = {
    "Credit card": CFPB_RAW / "credit_card.csv",
    "Checking or savings account":
        CFPB_RAW / "checking_or_savings_account.csv",
}

REQUIRED_COLUMNS = {
    "Date received",
    "Product",
    "Consumer complaint narrative",
}


class ArchiveLinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.current_href = None
        self.current_text = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            self.current_href = dict(attrs).get("href")
            self.current_text = []

    def handle_data(self, data):
        if self.current_href is not None:
            self.current_text.append(data)

    def handle_endtag(self, tag):
        if tag.lower() == "a" and self.current_href is not None:
            text = " ".join(" ".join(self.current_text).split())
            self.links.append((text, self.current_href))
            self.current_href = None
            self.current_text = []


def parse_received_date(value):
    value = (value or "").strip()
    if not value:
        return None

    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%m/%d/%y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass

    return None


def download_archives():
    headers = dict(UA)
    headers.setdefault("User-Agent", "ComplaintIQ-capstone/1.0")

    print("Reading official CFPB narrative archive:")
    print(ARCHIVE_PAGE)

    response = requests.get(
        ARCHIVE_PAGE,
        headers=headers,
        timeout=(20, 120),
    )
    response.raise_for_status()

    parser = ArchiveLinkParser()
    parser.feed(response.text)

    available_links = {
        " ".join(text.split()).casefold(): urljoin(ARCHIVE_PAGE, href)
        for text, href in parser.links
        if href
    }

    for label, filename in EXPORTS.items():
        url = available_links.get(label.casefold())

        if not url:
            raise RuntimeError(
                f"Could not find official archive link: {label}. "
                "Open the CFPB archive page and verify the export label."
            )

        destination = ARCHIVE_DIR / filename

        # Reuse a previously downloaded archive only if it is a valid ZIP.
        if destination.exists() and zipfile.is_zipfile(destination):
            print(f"Archive already present and valid: {destination.name}")
            continue

        print(f"Downloading: {label}")

        download = requests.get(
            url,
            headers=headers,
            timeout=(30, 300),
        )
        download.raise_for_status()

        if not zipfile.is_zipfile(io.BytesIO(download.content)):
            raise RuntimeError(
                f"The CFPB download for '{label}' was not a valid ZIP. "
                f"HTTP status: {download.status_code}; "
                f"content type: {download.headers.get('Content-Type')}"
            )

        temporary = destination.with_suffix(".zip.part")
        temporary.write_bytes(download.content)
        temporary.replace(destination)

        print(
            f"Saved {destination.name}: "
            f"{destination.stat().st_size:,} bytes"
        )


def process_archives():
    for output in PRODUCT_OUTPUTS.values():
        output.parent.mkdir(parents=True, exist_ok=True)
        if output.exists():
            output.unlink()

    counts = {product: 0 for product in PRODUCT_OUTPUTS}
    monthly_counts = {}

    for label, filename in EXPORTS.items():
        archive_path = ARCHIVE_DIR / filename

        if not archive_path.exists() or not zipfile.is_zipfile(archive_path):
            raise RuntimeError(f"Missing or invalid archive: {archive_path}")

        with zipfile.ZipFile(archive_path) as archive:
            csv_names = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv")
            ]

            if len(csv_names) != 1:
                raise RuntimeError(
                    f"Expected exactly one CSV in {filename}; "
                    f"found: {csv_names}"
                )

            csv_name = csv_names[0]
            print(f"\nProcessing {filename} -> {csv_name}")

            with archive.open(csv_name) as binary_file:
                text_file = io.TextIOWrapper(
                    binary_file,
                    encoding="utf-8-sig",
                    newline="",
                )
                reader = csv.DictReader(text_file)

                if not reader.fieldnames:
                    raise RuntimeError(f"No CSV header in {filename}")

                missing = REQUIRED_COLUMNS - set(reader.fieldnames)
                if missing:
                    raise RuntimeError(
                        f"{filename} is missing required columns: {missing}. "
                        f"Found: {reader.fieldnames}"
                    )

                # Open outputs in append mode; write the header only once.
                with (
                    PRODUCT_OUTPUTS["Credit card"].open(
                        "a", encoding="utf-8-sig", newline=""
                    ) as credit_file,
                    PRODUCT_OUTPUTS["Checking or savings account"].open(
                        "a", encoding="utf-8-sig", newline=""
                    ) as checking_file,
                ):
                    writers = {
                        "Credit card": csv.DictWriter(
                            credit_file, fieldnames=reader.fieldnames
                        ),
                        "Checking or savings account": csv.DictWriter(
                            checking_file, fieldnames=reader.fieldnames
                        ),
                    }

                    for product, writer in writers.items():
                        if counts[product] == 0:
                            writer.writeheader()

                    for row in reader:
                        received = parse_received_date(row.get("Date received"))
                        product = (row.get("Product") or "").strip()
                        narrative = (
                            row.get("Consumer complaint narrative") or ""
                        ).strip()

                        if received is None or not (
                            START_DATE <= received <= END_DATE
                        ):
                            continue

                        if product not in PRODUCT_OUTPUTS:
                            continue

                        if not narrative:
                            continue

                        writers[product].writerow(row)
                        counts[product] += 1
                        month_key = received.strftime("%Y-%m")
                        key = (filename, month_key, product)
                        monthly_counts[key] = monthly_counts.get(key, 0) + 1

    print("\n=== Output summary ===")
    for product, output in PRODUCT_OUTPUTS.items():
        print(f"{product}: {counts[product]:,} rows")
        print(f"  File: {output}")
        if not output.exists() or output.stat().st_size == 0:
            raise RuntimeError(f"Output file is empty or missing: {output}")

    print("\n=== Monthly counts by source archive ===")
    for (archive, month, product), count in sorted(monthly_counts.items()):
        print(f"{archive} | {month} | {product}: {count:,}")

    print("\nPASS: CFPB narrative archive processing completed.")


def main():
    download_archives()
    process_archives()


if __name__ == "__main__":
    main()
'''

script_path = PROJECT_ROOT / "src" / "download_data.py"
script_path.write_text(script, encoding="utf-8")
print("Wrote:", script_path)
print("Bytes:", script_path.stat().st_size)

Wrote: /home/sagemaker-user/complaintiq/src/download_data.py
Bytes: 8497


In [5]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "src.download_data"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=1800,
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"CFPB acquisition failed with exit code {result.returncode}. "
        "Review the output above before retrying."
    )

Reading official CFPB narrative archive:
https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/
Downloading: CCDB Export January 2026 through February 2026
Saved january_february_2026.zip: 36,832,036 bytes
Downloading: CCDB Export March 2026
Saved march_2026.zip: 22,053,643 bytes
Downloading: CCDB Export April 2026
Saved april_2026.zip: 22,963,936 bytes
Downloading: CCDB Export May 2026
Saved may_2026.zip: 21,033,928 bytes
Downloading: CCDB Export June 2026
Saved june_2026.zip: 19,624,767 bytes

Processing january_february_2026.zip -> CCDB_Export_15_January_2026_through_February_2026.csv

Processing march_2026.zip -> CCDB_Export_16_March_2026.csv

Processing april_2026.zip -> CCDB_Export_17_April_2026.csv

Processing may_2026.zip -> CCDB_Export_18_May_2026.csv

Processing june_2026.zip -> CCDB_Export_19_June_2026.csv

=== Output summary ===
Credit card: 18,613 rows
  File: /home/sagemaker-user/complaintiq/data/raw

In [6]:
import pandas as pd

outputs = {
    "Credit card": CFPB_RAW / "credit_card.csv",
    "Checking or savings account":
        CFPB_RAW / "checking_or_savings_account.csv",
}

for product, path in outputs.items():
    df = pd.read_csv(path)

    print(f"\n{product}")
    print("File:", path)
    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df.columns))
    print("Narrative column exists:",
          "Consumer complaint narrative" in df.columns)

    assert len(df) > 0
    assert "Consumer complaint narrative" in df.columns
    assert df["Consumer complaint narrative"].fillna("").str.strip().ne("").all()

    print("\nSample narrative:")
    print(df["Consumer complaint narrative"].iloc[0][:500])

print("\nPASS: Both CFPB datasets contain non-empty complaint narratives.")


Credit card
File: /home/sagemaker-user/complaintiq/data/raw/cfpb/credit_card.csv
Rows: 18,613
Columns: 16
Narrative column exists: True

Sample narrative:
Synchrony Bank Care Credit charged me {$1000.00} in interest XXXX XXXX I was making timely payments I had no warning this raise in interest was going occur I dont quite understand how the amount is compounded to my bill of {$1500.00}. Seems excessive. I can not pay my bill now as monthly payments have gone up. I have spoken to an online rep who said to call the XXXX line during business hours and they would make new arrangements. I was told I needed to make a {$900.00} payment for the interest c

Checking or savings account
File: /home/sagemaker-user/complaintiq/data/raw/cfpb/checking_or_savings_account.csv
Rows: 19,310
Columns: 16
Narrative column exists: True

Sample narrative:
On XXXXXXXX XXXX XXXX XXXX XXXX XXXXXXXX XXXX withdrew XXXX XXXX XXXX XXXX XXXX dollars from my checking account without giving me the ten-day prior notice

In [7]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

from src.config import ECFR_RAW, UA

BASE_URL = "https://www.ecfr.gov/api/versioner/v1"

# Regulatory parts selected for ComplaintIQ.
REGULATIONS = [
    {"title": 12, "part": "1002", "name": "Regulation B - Equal Credit Opportunity"},
    {"title": 12, "part": "1005", "name": "Regulation E - Electronic Fund Transfers"},
    {"title": 12, "part": "1006", "name": "Regulation F - Debt Collection"},
    {"title": 12, "part": "1022", "name": "Regulation V - Fair Credit Reporting"},
    {"title": 12, "part": "1026", "name": "Regulation Z - Truth in Lending"},
    {"title": 16, "part": "433", "name": "FTC Holder Rule"},
]

headers = dict(UA)
headers.setdefault("User-Agent", "ComplaintIQ-capstone/1.0")

session = requests.Session()
session.headers.update(headers)

# Discover the available publication date for each title.
response = session.get(f"{BASE_URL}/titles.json", timeout=60)
response.raise_for_status()
titles_payload = response.json()

title_metadata = {
    int(item["number"]): item
    for item in titles_payload["titles"]
}

download_manifest = []

for regulation in REGULATIONS:
    title = regulation["title"]
    part = regulation["part"]
    name = regulation["name"]

    metadata = title_metadata.get(title)
    if not metadata:
        raise RuntimeError(f"Title {title} missing from eCFR title list.")

    effective_date = (
        metadata.get("up_to_date_as_of")
        or metadata.get("latest_issue_date")
    )

    if not effective_date:
        raise RuntimeError(
            f"No available publication date found for Title {title}."
        )

    url = (
        f"{BASE_URL}/full/{effective_date}/title-{title}.xml"
    )

    print(f"Downloading Title {title}, Part {part}: {name}")
    print(f"Publication date: {effective_date}")

    response = session.get(
        url,
        params={"part": part},
        timeout=(30, 180),
    )
    response.raise_for_status()

    content = response.content

    # Validate that the returned document looks like XML.
    preview = content[:200].lstrip().lower()
    if not content or b"<" not in preview:
        raise RuntimeError(
            f"Unexpected XML response for Title {title}, Part {part}."
        )

    filename = f"title_{title}_part_{part}_{effective_date}.xml"
    destination = ECFR_RAW / filename
    destination.write_bytes(content)

    download_manifest.append({
        "title": title,
        "part": part,
        "name": name,
        "publication_date": effective_date,
        "url": response.url,
        "file": str(destination),
        "bytes": len(content),
        "sha256": __import__("hashlib").sha256(content).hexdigest(),
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    print(f"Saved: {destination.name} ({len(content):,} bytes)")
    time.sleep(1)

manifest_path = ECFR_RAW / "ecfr_manifest.json"
manifest_path.write_text(
    json.dumps(download_manifest, indent=2),
    encoding="utf-8",
)

print(f"\nManifest saved: {manifest_path}")
print(f"Regulatory XML files downloaded: {len(download_manifest)}")

Publication date: 2026-11-03
Saved: title_12_part_1002_2026-11-03.xml (504,474 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1005_2026-11-03.xml (859,423 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1006_2026-11-03.xml (231,338 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1022_2026-11-03.xml (272,782 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1026_2026-11-03.xml (3,946,980 bytes)
Publication date: 2026-11-03
Saved: title_16_part_433_2026-11-03.xml (6,582 bytes)

Manifest saved: /home/sagemaker-user/complaintiq/data/raw/ecfr/ecfr_manifest.json
Regulatory XML files downloaded: 6


In [8]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json

xml_files = sorted(ECFR_RAW.glob("title_*_part_*.xml"))

assert len(xml_files) == len(REGULATIONS), (
    f"Expected {len(REGULATIONS)} XML files; found {len(xml_files)}"
)

for path in xml_files:
    assert path.stat().st_size > 0, f"Empty file: {path}"

    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as exc:
        raise RuntimeError(f"Invalid XML in {path.name}: {exc}") from exc

    print(
        f"PASS: {path.name} | "
        f"{path.stat().st_size:,} bytes | "
        f"Root element: {root.tag}"
    )

manifest = json.loads(
    (ECFR_RAW / "ecfr_manifest.json").read_text(encoding="utf-8")
)

assert len(manifest) == len(REGULATIONS)
print("\nPASS: eCFR files are valid XML and the manifest is present.")

PASS: title_12_part_1002_2026-11-03.xml | 504,474 bytes | Root element: DIV5
PASS: title_12_part_1005_2026-11-03.xml | 859,423 bytes | Root element: DIV5
PASS: title_12_part_1006_2026-11-03.xml | 231,338 bytes | Root element: DIV5
PASS: title_12_part_1022_2026-11-03.xml | 272,782 bytes | Root element: DIV5
PASS: title_12_part_1026_2026-11-03.xml | 3,946,980 bytes | Root element: DIV5
PASS: title_16_part_433_2026-11-03.xml | 6,582 bytes | Root element: DIV5

PASS: eCFR files are valid XML and the manifest is present.


In [9]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json

xml_files = sorted(ECFR_RAW.glob("title_*_part_*.xml"))

assert len(xml_files) == len(REGULATIONS), (
    f"Expected {len(REGULATIONS)} XML files; found {len(xml_files)}"
)

for path in xml_files:
    assert path.stat().st_size > 0, f"Empty file: {path}"

    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as exc:
        raise RuntimeError(f"Invalid XML in {path.name}: {exc}") from exc

    print(
        f"PASS: {path.name} | "
        f"{path.stat().st_size:,} bytes | "
        f"Root element: {root.tag}"
    )

manifest = json.loads(
    (ECFR_RAW / "ecfr_manifest.json").read_text(encoding="utf-8")
)

assert len(manifest) == len(REGULATIONS)
print("\nPASS: eCFR files are valid XML and the manifest is present.")

PASS: title_12_part_1002_2026-11-03.xml | 504,474 bytes | Root element: DIV5
PASS: title_12_part_1005_2026-11-03.xml | 859,423 bytes | Root element: DIV5
PASS: title_12_part_1006_2026-11-03.xml | 231,338 bytes | Root element: DIV5
PASS: title_12_part_1022_2026-11-03.xml | 272,782 bytes | Root element: DIV5
PASS: title_12_part_1026_2026-11-03.xml | 3,946,980 bytes | Root element: DIV5
PASS: title_16_part_433_2026-11-03.xml | 6,582 bytes | Root element: DIV5

PASS: eCFR files are valid XML and the manifest is present.


In [3]:
from pathlib import Path

PDF_DIR = PROJECT_ROOT / "data" / "raw" / "pdf"

EXPECTED_PDFS = [
    "exam_manual_v2_2012.pdf",
    "sh_issue17_2018-09.pdf",
    "sh_issue18_2019-03.pdf",
    "sh_issue22_2020-09.pdf",
    "sh_issue23_2021-01.pdf",
]

print("PDF directory:", PDF_DIR)
print("=" * 70)

for filename in EXPECTED_PDFS:
    file_path = PDF_DIR / filename

    if file_path.exists():
        print(
            f"FOUND  | {filename} | "
            f"{file_path.stat().st_size:,} bytes"
        )
    else:
        print(f"MISSING | {filename}")

print("=" * 70)

found = sum((PDF_DIR / name).exists() for name in EXPECTED_PDFS)

print(f"Found: {found}/{len(EXPECTED_PDFS)}")
print(f"Missing: {len(EXPECTED_PDFS) - found}")

PDF directory: /home/sagemaker-user/complaintiq/data/raw/pdf
MISSING | exam_manual_v2_2012.pdf
MISSING | sh_issue17_2018-09.pdf
MISSING | sh_issue18_2019-03.pdf
MISSING | sh_issue22_2020-09.pdf
MISSING | sh_issue23_2021-01.pdf
Found: 0/5
Missing: 5


In [9]:
from pathlib import Path

PDF_DIR = PROJECT_ROOT / "data" / "raw" / "pdf"
PDF_DIR.mkdir(parents=True, exist_ok=True)

PDF_SOURCES = {
    "exam_manual_v2_2012.pdf": (
        "https://files.consumerfinance.gov/f/201210_cfpb_supervision-and-examination-manual-v2.pdf"
    ),
    "sh_issue17_2018-09.pdf": (
        "https://www.consumerfinance.gov/documents/6817/bcfp_supervisory-highlights_issue-17_2018-09.pdf"
    ),
    "sh_issue18_2019-03.pdf": (
        "https://files.consumerfinance.gov/f/documents/cfpb_supervisory-highlights_issue-18_032019.pdf"
    ),
    "sh_issue22_2020-09.pdf": (
        "https://www.consumerfinance.gov/documents/9164/cfpb_supervisory-highlights_issue-22_2020-09.pdf"
    ),
    "sh_issue23_2021-01.pdf": (
        "https://files.consumerfinance.gov/f/documents/cfpb-supervisory-highlights_issue-23_2021-01.pdf"
    ),
}

def is_valid_pdf(path):
    if not path.exists() or path.stat().st_size < 1000:
        return False
    with path.open("rb") as f:
        return f.read(5) == b"%PDF-"

for filename in PDF_SOURCES:
    path = PDF_DIR / filename
    if is_valid_pdf(path):
        print(f"VALID   | {filename} | {path.stat().st_size:,} bytes")
    elif path.exists():
        print(f"INVALID | {filename} | exists but is not a valid PDF")
    else:
        print(f"MISSING | {filename}")

print(f"\nPDF directory: {PDF_DIR}")

VALID   | exam_manual_v2_2012.pdf | 7,486,372 bytes
VALID   | sh_issue17_2018-09.pdf | 195,546 bytes
VALID   | sh_issue18_2019-03.pdf | 399,852 bytes
VALID   | sh_issue22_2020-09.pdf | 463,021 bytes
VALID   | sh_issue23_2021-01.pdf | 774,257 bytes

PDF directory: /home/sagemaker-user/complaintiq/data/raw/pdf


In [5]:
import requests
import time

session = requests.Session()
session.headers.update({
    "User-Agent": "ComplaintIQ Capstone Research Project/1.0 (joshua.paulv@ltm.com)"
})

download_results = []

for filename, url in PDF_SOURCES.items():
    destination = PDF_DIR / filename

    if is_valid_pdf(destination):
        print(f"SKIP    | Already valid: {filename}")
        download_results.append({
            "filename": filename,
            "status": "REUSED",
            "bytes": destination.stat().st_size,
            "url": url,
        })
        continue

    print(f"DOWNLOAD | {filename}")

    try:
        response = session.get(url, timeout=(20, 180))
        response.raise_for_status()

        content = response.content

        if len(content) < 1000 or not content.startswith(b"%PDF-"):
            raise ValueError(
                f"Response is not a valid PDF. "
                f"Content-Type={response.headers.get('Content-Type')}; "
                f"bytes={len(content)}"
            )

        temporary = destination.with_suffix(".pdf.part")
        temporary.write_bytes(content)

        # Validate temporary file before replacing destination.
        if not is_valid_pdf(temporary):
            temporary.unlink(missing_ok=True)
            raise ValueError("Downloaded file failed PDF signature validation.")

        temporary.replace(destination)

        print(f"SUCCESS  | {filename} | {len(content):,} bytes")

        download_results.append({
            "filename": filename,
            "status": "DOWNLOADED",
            "bytes": len(content),
            "url": url,
        })

        time.sleep(1)

    except Exception as exc:
        print(f"FAILED   | {filename} | {type(exc).__name__}: {exc}")
        download_results.append({
            "filename": filename,
            "status": "FAILED",
            "error": str(exc),
            "url": url,
        })

print("\nDownload summary")
for item in download_results:
    print(item["status"], "|", item["filename"])

failed = [item for item in download_results if item["status"] == "FAILED"]

if failed:
    print(f"\nSTOP: {len(failed)} PDF(s) failed. Do not mark Phase 1 complete.")
else:
    print("\nAll five PDF files are available.")

DOWNLOAD | exam_manual_v2_2012.pdf
SUCCESS  | exam_manual_v2_2012.pdf | 7,486,372 bytes
DOWNLOAD | sh_issue17_2018-09.pdf
SUCCESS  | sh_issue17_2018-09.pdf | 195,546 bytes
DOWNLOAD | sh_issue18_2019-03.pdf
SUCCESS  | sh_issue18_2019-03.pdf | 399,852 bytes
DOWNLOAD | sh_issue22_2020-09.pdf
SUCCESS  | sh_issue22_2020-09.pdf | 463,021 bytes
DOWNLOAD | sh_issue23_2021-01.pdf
FAILED   | sh_issue23_2021-01.pdf | HTTPError: 404 Client Error: Not Found for url: https://files.consumerfinance.gov/f/documents/cfpb-supervisory-highlights_issue-23_2021-01.pdf

Download summary
DOWNLOADED | exam_manual_v2_2012.pdf
DOWNLOADED | sh_issue17_2018-09.pdf
DOWNLOADED | sh_issue18_2019-03.pdf
DOWNLOADED | sh_issue22_2020-09.pdf
FAILED | sh_issue23_2021-01.pdf

STOP: 1 PDF(s) failed. Do not mark Phase 1 complete.
